<a href="https://colab.research.google.com/github/hoangmaituanhiep/DangerZoneDetection/blob/main/notebooks/colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!nvidia-smi
import torch
import tensorflow as tf
from google.colab import drive
drive.mount("/content/drive")

Thu Nov  5 01:14:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/DangerZoneDetection"
DATA_ZIP   = f"{DRIVE_ROOT}/data/FER2013.zip"
RUNS_DIR   = f"{DRIVE_ROOT}/runs"
MODELS_DIR = f"{DRIVE_ROOT}/models"
LOCAL_DATA = "/content/fer_plus"   # nơi giải nén (ổ cục bộ của Colab, đọc nhanh)

for d in (RUNS_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)

assert os.path.exists(DATA_ZIP), f"Không tìm thấy {DATA_ZIP}. Kiểm tra lại đã upload zip lên Drive chưa."

In [9]:
REPO     = "https://github.com/hoangmaituanhiep/DangerZoneDetection.git"
REPO_DIR = "/content/DangerZoneDetection"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO} {REPO_DIR}

%cd {REPO_DIR}

if not os.path.exists(LOCAL_DATA):
    !cp "{DATA_ZIP}" /content/FER2013.zip
    !unzip -q /content/FER2013.zip -d {LOCAL_DATA}
else:
    !rm -rf /content/fer_plus/
    !cp "{DATA_ZIP}" /content/FER2013.zip
    !unzip -q /content/FER2013.zip -d {LOCAL_DATA}

!ls {LOCAL_DATA}

Already up to date.
/content/DangerZoneDetection
FER2013Test  FER2013Train  FER2013Valid


In [10]:
import glob
import pandas as pd

train_dirs = glob.glob(f"{LOCAL_DATA}/**/FER2013Train", recursive=True)
assert train_dirs, "Không thấy thư mục FER2013Train trong dữ liệu đã giải nén."
BASE = os.path.dirname(train_dirs[0])
print("Thư mục gốc dữ liệu:", BASE)

SPLITS = {"train": "FER2013Train", "val": "FER2013Valid", "test": "FER2013Test"}
EMOTIONS = ["neutral", "happiness", "surprise", "sadness", "anger", "disgust", "fear", "contempt"]
VOTE_COLS = EMOTIONS + ["unknown", "NF"]

def load_split(folder):
    # label.csv: tên ảnh, khung mặt, rồi 10 cột số phiếu (không có dòng tiêu đề)
    df = pd.read_csv(f"{BASE}/{folder}/label.csv", header=None)
    df.columns = ["image", "box"] + VOTE_COLS
    votes = df[VOTE_COLS].values
    top = votes.argmax(axis=1)
    df["label_id"] = top
    df = df[df["label_id"] < len(EMOTIONS)].copy()          # bỏ unknown / NF
    df["label"] = df["label_id"].map(lambda i: EMOTIONS[i])
    df["path"] = f"{BASE}/{folder}/" + df["image"]
    return df.reset_index(drop=True)

dfs = {name: load_split(folder) for name, folder in SPLITS.items()}

for name, df in dfs.items():
    print(f"\n== {name}: {len(df)} ảnh ==")
    print(df["label"].value_counts().to_string())

# kiểm tra file ảnh có thật
missing = [p for p in dfs["train"]["path"].head(200) if not os.path.exists(p)]
print("\nẢnh bị thiếu (trong 200 ảnh đầu của train):", len(missing))

Thư mục gốc dữ liệu: /content/fer_plus

== train: 28385 ảnh ==
label
neutral      10295
happiness     7526
surprise      3557
sadness       3530
anger         2463
fear           655
disgust        191
contempt       168

== val: 3555 ảnh ==
label
neutral      1329
happiness     899
surprise      458
sadness       415
anger         325
fear           71
disgust        32
contempt       26

== test: 3545 ảnh ==
label
neutral      1258
happiness     928
surprise      447
sadness       446
anger         321
fear           97
contempt       28
disgust        20

Ảnh bị thiếu (trong 200 ảnh đầu của train): 0
